In [9]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from math import sqrt
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# ── AUTO-PRONALAŽENJE FOLDERA SA CSV-ima ──────────────────────────────────
possible_dirs = ["data", ".", "./data", "../data"]
INPUT_DIR = None

for dir_path in possible_dirs:
    if os.path.exists(dir_path):
        csv_files = glob.glob(os.path.join(dir_path, "*_Data.csv"))
        if csv_files:
            INPUT_DIR = dir_path
            break

if INPUT_DIR is None:
    print("❌ GREŠKA: Nema CSV fajlova pronađenih!")
    print("Kreiraj 'data' folder i stavi CSV fajlove tamo (npr. EURUSD_Data.csv, USDCNY_Data.csv...)")
    INPUT_DIR = "data"  # default fallback
else:
    print(f"✓ CSV fajlovi pronađeni u folderu: {INPUT_DIR}")

ID_COLUMN = "Pair"
print("✓ Sve biblioteke uspešno učitane!")
print(f"Trenutni radni folder: {os.getcwd()}")

✓ CSV fajlovi pronađeni u folderu: data
✓ Sve biblioteke uspešno učitane!
Trenutni radni folder: /Users/djordjeradosavljevic/Downloads/fax/N/Projekat/v_p


In [10]:
def clean_usdcny_csv(input_dir="."):
    """Uklanja redove koji odstupaju od standardnog broja kolona (anomalije u formatu)"""
    usdcny_path = os.path.join(input_dir, "USDCNY_Data.csv")
    if not os.path.exists(usdcny_path):
        print("⚠️ USDCNY_Data.csv nije pronađen, preskačem namensko čišćenje.")
        return False

    with open(usdcny_path, 'r', encoding='utf-8-sig') as f:
        lines = f.readlines()

    header = lines[0].strip()
    expected_cols = len(header.split(','))

    cleaned_lines = [lines[0]]
    removed_count = 0

    for i, line in enumerate(lines[1:], 1):
        cols = len(line.strip().split(','))
        if cols != expected_cols:
            removed_count += 1
        else:
            cleaned_lines.append(line)

    with open(usdcny_path, 'w', encoding='utf-8-sig') as f:
        f.writelines(cleaned_lines)

    print(f"✓ USDCNY_Data.csv očišćen! Uklonjeno problematičnih redova: {removed_count}")
    return True

clean_usdcny_csv(INPUT_DIR)

True

In [11]:
def build_currency_data_all(input_dir: str) -> pd.DataFrame:
    """Učitava sve pojedinačne parove, izvlači Date i Price i pakuje ih u 'long' format (redove)"""
    csv_files = sorted(glob.glob(os.path.join(input_dir, "*_Data.csv")))
    if not csv_files:
        raise FileNotFoundError(f"Nije pronađen nijedan *_Data.csv fajl u folderu: {input_dir}")

    print(f"\nSpajanje podataka iz {len(csv_files)} fajlova u jedan zajednički hronološki niz...")
    frames = []
    for path in csv_files:
        fname = os.path.basename(path)
        pair_name = fname.replace("_Data.csv", "")

        df = pd.read_csv(path, encoding="utf-8-sig")
        df.columns = [c.strip().strip('"') for c in df.columns]

        df = df[["Date", "Price"]].copy()
        df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
        df["Price"] = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce")
        df["Pair"] = pair_name

        frames.append(df[["Pair", "Date", "Price"]])

    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["Pair", "Date"]).reset_index(drop=True)
    return result

def analyze_missing_values(currency_all: pd.DataFrame) -> pd.DataFrame:
    """Računa ukupan broj i procentualni udeo nedostajućih vrednosti po svakom valutnom paru"""
    rows = []
    for pair, group in currency_all.groupby("Pair"):
        total_rows = len(group)
        missing = group["Price"].isna().sum()
        pct = (missing / total_rows * 100) if total_rows > 0 else 0
        rows.append({
            "Pair": pair,
            "Total_Rows": total_rows,
            "Missing_Count": missing,
            "Missing_Pct": round(pct, 4)
        })
    return pd.DataFrame(rows).sort_values("Pair").reset_index(drop=True)

# Izgradnja i pokretanje analize
currency_all = build_currency_data_all(INPUT_DIR)
missing_summary = analyze_missing_values(currency_all)

print("\n📊 ANALIZA NEDOSTAJUĆIH VRIJEDNOSTI PO PAROVIMA:")
print(missing_summary.to_string(index=False))


Spajanje podataka iz 6 fajlova u jedan zajednički hronološki niz...

📊 ANALIZA NEDOSTAJUĆIH VRIJEDNOSTI PO PAROVIMA:
  Pair  Total_Rows  Missing_Count  Missing_Pct
EURUSD        2610              0          0.0
USDCAD        2610              0          0.0
USDCHF        2610              0          0.0
USDCNY        2611              0          0.0
USDGBP        2610              0          0.0
USDJPY        2610              0          0.0


In [12]:
def build_currency_data_all(input_dir: str) -> pd.DataFrame:
    """Učitava sve pojedinačne parove, izvlači Date i Price i pakuje ih u 'long' format (redove)"""
    csv_files = sorted(glob.glob(os.path.join(input_dir, "*_Data.csv")))
    if not csv_files:
        raise FileNotFoundError(f"Nije pronađen nijedan *_Data.csv fajl u folderu: {input_dir}")

    print(f"\nSpajanje podataka iz {len(csv_files)} fajlova u jedan zajednički hronološki niz...")
    frames = []
    for path in csv_files:
        fname = os.path.basename(path)
        pair_name = fname.replace("_Data.csv", "")

        df = pd.read_csv(path, encoding="utf-8-sig")
        df.columns = [c.strip().strip('"') for c in df.columns]

        df = df[["Date", "Price"]].copy()
        df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
        df["Price"] = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce")
        df["Pair"] = pair_name

        frames.append(df[["Pair", "Date", "Price"]])

    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["Pair", "Date"]).reset_index(drop=True)
    return result

def analyze_missing_values(currency_all: pd.DataFrame) -> pd.DataFrame:
    """Računa ukupan broj i procentualni udeo nedostajućih vrednosti po svakom valutnom paru"""
    rows = []
    for pair, group in currency_all.groupby("Pair"):
        total_rows = len(group)
        missing = group["Price"].isna().sum()
        pct = (missing / total_rows * 100) if total_rows > 0 else 0
        rows.append({
            "Pair": pair,
            "Total_Rows": total_rows,
            "Missing_Count": missing,
            "Missing_Pct": round(pct, 4)
        })
    return pd.DataFrame(rows).sort_values("Pair").reset_index(drop=True)

# Izgradnja i pokretanje analize
currency_all = build_currency_data_all(INPUT_DIR)
missing_summary = analyze_missing_values(currency_all)

print("\n📊 ANALIZA NEDOSTAJUĆIH VRIJEDNOSTI PO PAROVIMA:")
print(missing_summary.to_string(index=False))


Spajanje podataka iz 6 fajlova u jedan zajednički hronološki niz...

📊 ANALIZA NEDOSTAJUĆIH VRIJEDNOSTI PO PAROVIMA:
  Pair  Total_Rows  Missing_Count  Missing_Pct
EURUSD        2610              0          0.0
USDCAD        2610              0          0.0
USDCHF        2610              0          0.0
USDCNY        2611              0          0.0
USDGBP        2610              0          0.0
USDJPY        2610              0          0.0


In [13]:
def _sma(series, period):
    return series.rolling(window=period, min_periods=1).mean()

def _ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def _kama(series, period=14, fast=2, slow=30):
    n = len(series)
    kama_vals = np.full(n, np.nan)
    sc_fast, sc_slow = 2 / (fast + 1), 2 / (slow + 1)
    prices = series.values
    first_valid = 0
    while first_valid < n and np.isnan(prices[first_valid]): first_valid += 1
    if first_valid >= n: return pd.Series(kama_vals, index=series.index)

    kama_vals[first_valid] = prices[first_valid]
    for i in range(first_valid + 1, n):
        if i < period:
            kama_vals[i] = prices[i]
            continue
        direction = abs(prices[i] - prices[i - period])
        volatility = np.nansum(np.abs(np.diff(prices[i - period: i + 1])))
        er = direction / volatility if volatility != 0 else 0
        sc = (er * (sc_fast - sc_slow) + sc_slow) ** 2
        kama_vals[i] = kama_vals[i - 1] + sc * (prices[i] - kama_vals[i - 1])
    return pd.Series(kama_vals, index=series.index)

def _macd(series, fast, slow, signal):
    ema_fast = _ema(series, fast)
    ema_slow = _ema(series, slow)
    macd_line = ema_fast - ema_slow
    signal_line = _ema(macd_line, signal)
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

def _roc(series, period):
    return series.pct_change(periods=period) * 100

def _sar(high, low, af_start=0.02, af_max=0.2):
    sar = np.full(len(high), np.nan)
    trend, sar[0], ep, af = 1, low.iloc[0], high.iloc[0], af_start
    for i in range(1, len(high)):
        prev_sar = sar[i - 1]
        if trend == 1:
            sar[i] = prev_sar + af * (ep - prev_sar)
            sar[i] = min(sar[i], low.iloc[i - 1], low.iloc[i - 2] if i >= 2 else low.iloc[i - 1])
            if low.iloc[i] < sar[i]:
                trend, sar[i], ep, af = -1, ep, low.iloc[i], af_start
            else:
                if high.iloc[i] > ep: ep, af = high.iloc[i], min(af + af_start, af_max)
        else:
            sar[i] = prev_sar - af * (prev_sar - ep)
            sar[i] = max(sar[i], high.iloc[i - 1], high.iloc[i - 2] if i >= 2 else high.iloc[i - 1])
            if high.iloc[i] > sar[i]:
                trend, sar[i], ep, af = 1, ep, high.iloc[i], af_start
            else:
                if low.iloc[i] < ep: ep, af = low.iloc[i], min(af + af_start, af_max)
    return pd.Series(sar, index=high.index)

def _cci(high, low, close, period):
    typical = (high + low + close) / 3
    sma_tp = typical.rolling(period, min_periods=1).mean()
    mad = typical.rolling(period, min_periods=1).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    return (typical - sma_tp) / (0.015 * mad)

def _rsi(series, period):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period, min_periods=1).mean()
    loss = (-delta.clip(upper=0)).rolling(period, min_periods=1).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def _cmo(series, period):
    delta = series.diff()
    up = delta.clip(lower=0).rolling(period, min_periods=1).sum()
    down = (-delta.clip(upper=0)).rolling(period, min_periods=1).sum()
    return ((up - down) / (up + down).replace(0, np.nan)) * 100

def _rci(series, period):
    def _rci_single(window):
        n = len(window)
        price_rank = pd.Series(window).rank(ascending=False).values
        time_rank = np.arange(1, n + 1)
        d2 = np.sum((price_rank - time_rank) ** 2)
        return (1 - 6 * d2 / (n * (n**2 - 1))) * 100
    return series.rolling(period, min_periods=period).apply(_rci_single, raw=True)

def _std(series, period):
    return series.rolling(period, min_periods=1).std()

print("✓ Matematičke funkcije za sve tehničke indikatore uspešno učitane.")

✓ Matematičke funkcije za sve tehničke indikatore uspešno učitane.


In [14]:
PARAMS = {
    "SMA_period"  : 14, "EMA_period"  : 14, "KAMA_period" : 14,
    "MACD_fast"   : 12, "MACD_slow"   : 26, "MACD_signal" : 9,
    "ROC_period"  : 14, "SAR_af"      : 0.025, "SAR_max_af": 0.25,
    "CCI_period"  : 14, "RSI_period"  : 14, "CMO_period"  : 14,
    "RCI_period"  : 14, "STD_period"  : 14,
    "SMA_of_ind"  : 9,  # SMA za indikatore 5-11
    "ROC_of_ind"  : 9,  # ROC za indikatore 1-11
    "SMA_output"  : 9, "ROC_output"  : 9
}

def compute_indicators(price_series: pd.Series, p: dict) -> pd.DataFrame:
    """Računanje 11 baznih indikatora, SMA(5-11) i ROC(1-11) prema specifikaciji"""
    close = price_series.copy().astype(float)
    high, low = close, close
    cols = {}

    # Indikatori 1-4
    cols["SMA"] = _sma(close, p["SMA_period"])
    cols["EMA"] = _ema(close, p["EMA_period"])
    cols["KAMA"] = _kama(close, p["KAMA_period"])
    macd_line, _, _ = _macd(close, p["MACD_fast"], p["MACD_slow"], p["MACD_signal"])
    cols["MACD"] = macd_line

    # Indikatori 5-11
    ind_5_11 = {
        "ROC": _roc(close, p["ROC_period"]), "SAR": _sar(high, low, p["SAR_af"], p["SAR_max_af"]),
        "CCI": _cci(high, low, close, p["CCI_period"]), "RSI": _rsi(close, p["RSI_period"]),
        "CMO": _cmo(close, p["CMO_period"]), "RCI": _rci(close, p["RCI_period"]), "STD": _std(close, p["STD_period"])
    }
    for name, series in ind_5_11.items():
        cols[name] = series

    # Dodatno izračunavanje: SMA od vrednosti indikatora (Samo za 5-11)
    for name in ind_5_11.keys():
        cols[f"{name}_SMA"] = _sma(cols[name], p["SMA_of_ind"])

    # Dodatno izračunavanje: ROC od vrednosti indikatora (Za sve od 1-11)
    bazni_1_11 = ["SMA", "EMA", "KAMA", "MACD", "ROC", "SAR", "CCI", "RSI", "CMO", "RCI", "STD"]
    for name in bazni_1_11:
        cols[f"{name}_ROC"] = _roc(cols[name], p["ROC_of_ind"])

    return pd.DataFrame(cols, index=close.index)

def compute_outputs(price_series: pd.Series, p: dict) -> pd.DataFrame:
    """Priprema izlaza: Price, ROC(Price), SMA(Price), SMA(ROC(Price))"""
    close = price_series.copy().astype(float)
    price_roc = _roc(close, p["ROC_output"])
    return pd.DataFrame({
        "Price": close,
        "Price_ROC": price_roc,
        "Price_SMA": _sma(close, p["SMA_output"]),
        "Price_ROC_SMA": _sma(price_roc, p["SMA_output"])
    }, index=close.index)

def process_all_series(data_all: pd.DataFrame, id_column: str, params: dict) -> pd.DataFrame:
    all_frames = []
    for item_id, sub in data_all.groupby(id_column):
        sub = sub.sort_values("Date").dropna(subset=["Price"]).reset_index(drop=True)
        if len(sub) < 50: continue

        price_series = sub.set_index("Date")["Price"]
        ind_df = compute_indicators(price_series, params)
        out_df = compute_outputs(price_series, params)

        combined = ind_df.join(out_df).reset_index()
        combined.insert(0, id_column, item_id)
        all_frames.append(combined)
    return pd.concat(all_frames, ignore_index=True)

print("Izračunavam sve ulazne i izlazne promenljive za sve parove...")
features = process_all_series(currency_all, ID_COLUMN, PARAMS)
print(f"✓ Završeno! Ukupna matrica strukture: {features.shape}")

✓ Završeno! Ukupna matrica strukture: (15661, 35)


In [15]:
N_PRICE_LAGS = 5
TEST_SIZE = 0.2
EXP_3_3_AS_REGRESSION = False  # False = Klasifikacija smera, True = Regresija prinosa (iz teksta)

MLP_PARAMS = dict(
    hidden_layer_sizes=(32, 16), activation="relu", solver="adam",
    max_iter=500, early_stopping=True, n_iter_no_change=15, random_state=42
)

HORIZONS_1_1 = [1, 5, 10]
HORIZONS_OTHER = [1, 3, 5, 10]

INDICATOR_BASE_COLS = ["SMA", "EMA", "KAMA", "MACD", "ROC", "SAR", "CCI", "RSI", "CMO", "RCI", "STD"]
INDICATOR_SMA_COLS  = [f"{c}_SMA" for c in ["ROC", "SAR", "CCI", "RSI", "CMO", "RCI", "STD"]]
INDICATOR_ROC_COLS  = [f"{c}_ROC" for c in INDICATOR_BASE_COLS]

def build_experiment_definitions():
    exps = {}
    # BLOK 1: Predviđanje cena
    exps["1-1a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["price"])
    exps["1-1b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["indicators"])
    exps["1-1c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_1_1, input_groups=["price", "indicators"])
    exps["1-2a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price"])
    exps["1-2b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators"])
    exps["1-2c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "indicators", "roc_price", "roc_indicators"])
    exps["1-3a"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
    exps["1-3b"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
    exps["1-3c"] = dict(problem="regression", target_type="price", horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    # BLOK 2: Predviđanje prinosa
    for sub, groups in {"1": (["price"], ["indicators"], ["price", "indicators"]),
                        "2": (["price", "roc_price"], ["indicators", "roc_indicators"], ["price", "indicators", "roc_price", "roc_indicators"]),
                        "3": (["price", "roc_price", "sma_price"], ["indicators", "roc_indicators", "sma_indicators"], ["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])}.items():
        exps[f"2-{sub}a"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=groups[0])
        exps[f"2-{sub}b"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=groups[1])
        exps[f"2-{sub}c"] = dict(problem="regression", target_type="return", horizons=HORIZONS_OTHER, input_groups=groups[2])

    # BLOK 3: Predviđanje trenda / smera
    for sub, groups in {"1": (["price"], ["indicators"], ["price", "indicators"]),
                        "2": (["price", "roc_price"], ["indicators", "roc_indicators"], ["price", "indicators", "roc_price", "roc_indicators"])}.items():
        exps[f"3-{sub}a"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=groups[0])
        exps[f"3-{sub}b"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=groups[1])
        exps[f"3-{sub}c"] = dict(problem="classification", target_type="sign", horizons=HORIZONS_OTHER, input_groups=groups[2])

    # Eksperiment 3-3 (sa uslovnim rešenjem)
    prob_33 = "regression" if EXP_3_3_AS_REGRESSION else "classification"
    targ_33 = "return" if EXP_3_3_AS_REGRESSION else "sign"
    exps["3-3a"] = dict(problem=prob_33, target_type=targ_33, horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price"])
    exps["3-3b"] = dict(problem=prob_33, target_type=targ_33, horizons=HORIZONS_OTHER, input_groups=["indicators", "roc_indicators", "sma_indicators"])
    exps["3-3c"] = dict(problem=prob_33, target_type=targ_33, horizons=HORIZONS_OTHER, input_groups=["price", "roc_price", "sma_price", "indicators", "roc_indicators", "sma_indicators"])

    return exps

print("✓ Arhitektura eksperimenata uspešno mapirana.")

✓ Arhitektura eksperimenata uspešno mapirana.


In [16]:
def add_price_lags(df, n_lags=N_PRICE_LAGS):
    df = df.copy()
    for lag in range(1, n_lags + 1): df[f"Price_lag{lag}"] = df["Price"].shift(lag)
    return df

def add_targets(df, horizons):
    df = df.copy()
    for h in horizons:
        future_price = df["Price"].shift(-h)
        df[f"Target_Price_t{h}"] = future_price
        df[f"Target_Return_t{h}"] = (future_price - df["Price"]) / df["Price"] * 100
        df[f"Target_Sign_t{h}"] = (df[f"Target_Return_t{h}"] > 0).astype(int)
    return df

def get_feature_groups(df_columns):
    cols = set(df_columns)
    price_cols = ["Price"] + [f"Price_lag{i}" for i in range(1, N_PRICE_LAGS + 1)]
    return {
        "price": [c for c in price_cols if c in cols],
        "roc_price": [c for c in ["Price_ROC"] if c in cols],
        "sma_price": [c for c in ["Price_SMA", "Price_ROC_SMA"] if c in cols],
        "indicators": [c for c in INDICATOR_BASE_COLS if c in cols],
        "roc_indicators": [c for c in INDICATOR_ROC_COLS if c in cols],
        "sma_indicators": [c for c in INDICATOR_SMA_COLS if c in cols]
    }

def combine_groups(feature_groups, group_names):
    cols = []
    for g in group_names: cols.extend(feature_groups.get(g, []))
    return list(set(cols))

In [17]:
def run_mlp(df, feature_cols, target_col, problem, mlp_params=MLP_PARAMS, test_size=TEST_SIZE):
    data = df[feature_cols + [target_col]].dropna()
    if len(data) < 50: return None

    X, y = data[feature_cols].values, data[target_col].values

    # Hronološki split bez mešanja (Time-Series Split)
    split = int(len(data) * (1 - test_size))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    if problem == "regression":
        model = MLPRegressor(**mlp_params)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
        return {
            "MAE": mean_absolute_error(y_test, preds), "RMSE": sqrt(mean_squared_error(y_test, preds)), "R2": r2_score(y_test, preds),
            "n_train": len(y_train), "n_test": len(y_test), "n_features": len(feature_cols)
        }
    else:
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2: return None
        model = MLPClassifier(**mlp_params)
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
        return {
            "Accuracy": accuracy_score(y_test, preds), "Precision": precision_score(y_test, preds, zero_division=0),
            "Recall": recall_score(y_test, preds, zero_division=0), "F1": f1_score(y_test, preds, zero_division=0),
            "n_train": len(y_train), "n_test": len(y_test), "n_features": len(feature_cols)
        }

def run_all_experiments_for_series(df_series, series_id, id_column, experiments):
    results = []
    df = df_series.sort_values("Date").reset_index(drop=True)
    df = add_price_lags(df, N_PRICE_LAGS)